# Topic modeling: LDA baseline and BERTopic

**Owner:** Ibraheem

Skeleton only: fill in the empty code cells; do not change contracts.

In [ ]:
# Data paths: run this cell first.
import sys
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
DATA_PATH = REPO_DIR / "data" / "papers.parquet"
EMBEDDINGS_PATH = REPO_DIR / "data" / "embeddings.npy"
DATA_PATH, EMBEDDINGS_PATH


## Load cleaned abstracts

Load the lemmatized text from data/papers.parquet.

In [ ]:
import numpy as np
import pandas as pd

papers = pd.read_parquet(DATA_PATH, columns=["id", "title", "abstract", "text_lemma"])
texts_lemma = papers["text_lemma"].astype(str).tolist()
tokenized_texts = [t.split() for t in texts_lemma]
print(f"papers: {len(papers):,}")
print(f"columns: {list(papers.columns)}")
print(f"first title: {papers['title'].iloc[0][:100]}")


## Tune LDA topic count

Compare c_v coherence for several k values.

In [ ]:
# k=20 is fixed by the experiment specification. No k sweep is run: a
# full-corpus sweep would multiply the training cost, and a sampled sweep
# would silently subsample. c_v is reported for this model below.
K = 20
print(f"Using spec-fixed k={K}; c_v is computed after fitting on all papers.")


## Fit the final LDA model

Train the selected k-topic model on all papers.

In [ ]:
import time
from src.topics import run_lda

t0 = time.perf_counter()
lda = run_lda(texts_lemma, k=K, seed=42)
print(f"LDA runtime: {time.perf_counter() - t0:.1f}s")
print(f"topics: {lda['k']}, model: {lda['model_path']}")
for t, words in enumerate(lda["topic_words"]):
    print(f"LDA topic {t}: {', '.join(words)}")


## Fit BERTopic

Use the Kaggle embeddings with BERTopic and reduce topics.

In [ ]:
from src.topics import run_bertopic

embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
print(f"embeddings shape: {embeddings.shape}")
t0 = time.perf_counter()
bt = run_bertopic(texts_lemma, embeddings, nr_topics=25)
print(f"BERTopic runtime: {time.perf_counter() - t0:.1f}s")
print(f"topics: {bt['nr_topics']}, outlier share: {bt['outlier_share']:.4f}")
for tid, words in zip(bt["topic_ids"], bt["topic_words"]):
    print(f"BERTopic {tid}: {', '.join(words)}")


## Label topics

Turn top words into short human-readable labels.

In [ ]:
from IPython.display import Markdown, display
from src.topics import coherence_cv, label_topics

lda_c_v = coherence_cv(lda["topic_words"], tokenized_texts)
bt_c_v = coherence_cv(bt["topic_words"], tokenized_texts)
print(f"LDA c_v: {lda_c_v:.4f}")
print(f"BERTopic c_v: {bt_c_v:.4f}")

lda_labels = label_topics(lda["topic_words"])
bt_label_map = label_topics(bt["topic_words"])
bt_labels = {tid: bt_label_map[i] for i, tid in enumerate(bt["topic_ids"])}
bt_labels[-1] = "Outliers"

display(Markdown(
    "| Model | c_v | topics | outlier share |\n"
    "|---|---|---|---|\n"
    f"| LDA | {lda_c_v:.4f} | {lda['k']} | 0.0000 |\n"
    f"| BERTopic | {bt_c_v:.4f} | {bt['nr_topics']} | {bt['outlier_share']:.4f} |"
))

lda_md = ["| topic | label | top words |", "|---|---|---|"]
for t in range(lda["k"]):
    lda_md.append(f"| {t} | {lda_labels[t]} | {', '.join(lda['topic_words'][t][:5])} |")
display(Markdown("### LDA labels\n" + "\n".join(lda_md)))

bt_md = ["| topic | label | top words |", "|---|---|---|"]
for tid in bt["topic_ids"]:
    i = bt["topic_ids"].index(tid)
    bt_md.append(f"| {tid} | {bt_labels[tid]} | {', '.join(bt['topic_words'][i][:5])} |")
display(Markdown("### BERTopic labels\n" + "\n".join(bt_md)))

lda_arr = np.asarray(lda["doc_topics"])
bt_arr = np.asarray(bt["doc_topics"])
for name, arr, labels in (("LDA", lda_arr, lda_labels), ("BERTopic", bt_arr, bt_labels)):
    print(f"\n3 sample documents for 5 {name} topics:")
    for tid in list(labels)[:5]:
        idxs = np.where(arr == tid)[0][:3]
        print(f"  {name} topic {tid} ({labels[tid]}):")
        for j in idxs:
            print(f"    - {papers['title'].iloc[int(j)][:110]}")


## Save assignments

Write data/doc_topics.parquet for the trend notebook.

In [ ]:
import json
from src.settings_p2 import DOC_TOPICS_PATH, ROOT

assign = pd.DataFrame({
    "id": papers["id"].to_numpy(),
    "lda_topic": np.asarray(lda["doc_topics"], dtype=np.int64),
    "bertopic_topic": np.asarray(bt["doc_topics"], dtype=np.int64),
})
assert len(assign) == len(papers) and assign["id"].equals(papers["id"].reset_index(drop=True))
assign.to_parquet(DOC_TOPICS_PATH, index=False)
print(f"wrote {DOC_TOPICS_PATH} ({len(assign):,} rows)")

lda_json = {
    "model": "gensim LDA", "k": lda["k"], "seed": lda["seed"],
    "c_v": round(lda_c_v, 4), "outlier_share": 0.0,
    "topics": [
        {"topic_id": t, "label": lda_labels[t], "top_words": lda["topic_words"][t]}
        for t in range(lda["k"])
    ],
}
bt_json = {
    "model": "BERTopic", "nr_topics": bt["nr_topics"], "nr_topics_requested": 25,
    "c_v": round(bt_c_v, 4), "outlier_share": round(bt["outlier_share"], 4),
    "topics": [
        {"topic_id": tid, "label": bt_labels[tid], "top_words": bt["topic_words"][i]}
        for i, tid in enumerate(bt["topic_ids"])
    ],
}
for name, payload in (("topics_lda.json", lda_json), ("topics_bertopic.json", bt_json)):
    (ROOT / "results" / name).write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"wrote results/{name}")


## Results

TODO: summarize the outputs and figures produced above.